# 5. 다양한 RAG 구성 비교하기

이 노트북은 BM25, Vector Search, Hybrid Search를 실제로 구축하고 같은 평가 질문으로 비교합니다.

비교 대상은 다음 세 가지입니다.

- A: BM25
- B: Vector Search
- C: Hybrid Search


## 비교 원칙

검색기만 비교하려면 나머지 조건을 고정해야 합니다.

| 고정할 조건 | 이유 |
| --- | --- |
| 문서와 chunk 설정 | 문서 분할이 달라지면 검색기 차이를 분리하기 어려움 |
| 평가 질문과 기준 정답 | 같은 문제를 풀어야 비교 가능 |
| 답변 생성 모델과 프롬프트 | 생성 단계 차이를 최소화 |
| RAGAS 평가 지표와 평가 모델 | 점수 기준을 동일하게 유지 |

이 노트북에서는 같은 chunk와 같은 프롬프트를 사용하고 검색기만 바꿉니다.


In [ ]:
from __future__ import annotations

import ast
import json
import math
import re
import shutil
from collections import Counter
from pathlib import Path

import pandas as pd
from datasets import Dataset
from dotenv import load_dotenv
from IPython.display import display
from langchain_chroma import Chroma
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ragas import evaluate
from ragas.metrics import answer_relevancy, context_precision, context_recall, faithfulness

load_dotenv()

pd.set_option("display.max_colwidth", 120)

DATA_DIR = Path("./data")
PDF_PATH = DATA_DIR / "SPRI_AI_Brief_2023년12월호_F.pdf"
QUESTIONS_PATH = DATA_DIR / "evaluation_questions_for_ragas.csv"

METRICS = [
    "answer_relevancy",
    "faithfulness",
    "context_recall",
    "context_precision",
]


In [ ]:
def parse_context_list(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return [str(value)]
    return parsed if isinstance(parsed, list) else [str(parsed)]


questions = pd.read_csv(QUESTIONS_PATH)
questions["reference_contexts"] = questions["reference_contexts"].apply(parse_context_list)

# 수업 시간과 API 비용을 조절하려면 숫자를 줄입니다. 전체 평가를 하려면 None으로 바꿉니다.
MAX_EVAL_SAMPLES = 15
eval_questions = questions.head(MAX_EVAL_SAMPLES).copy() if MAX_EVAL_SAMPLES else questions.copy()

print(f"평가 질문 수: {len(eval_questions)}")
display(eval_questions[["user_input", "query_style", "query_category", "reference"]].head())


## 공통 문서 분할과 Vector DB 구축

BM25, Vector Search, Hybrid Search 모두 같은 chunk를 사용합니다.


In [ ]:
documents = PDFPlumberLoader(str(PDF_PATH)).load()

COMMON_CHUNK_SIZE = 500
COMMON_CHUNK_OVERLAP = 50
RETRIEVER_K = 4
HYBRID_CANDIDATES = 8

splitter = RecursiveCharacterTextSplitter(
    chunk_size=COMMON_CHUNK_SIZE,
    chunk_overlap=COMMON_CHUNK_OVERLAP,
)
split_documents = splitter.split_documents(documents)

print(f"PDF 페이지 수: {len(documents)}")
print(f"분할 문서 수: {len(split_documents)}")


In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

CHROMA_PATH = Path("./chroma_retriever_comparison")
COLLECTION_NAME = "retriever_comparison"

if CHROMA_PATH.exists():
    shutil.rmtree(CHROMA_PATH)

vectorstore = Chroma.from_documents(
    documents=split_documents,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=str(CHROMA_PATH),
)

answer_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


## BM25 검색기 구현

아래 BM25는 강의용으로 직접 구현한 검색기입니다. 문서를 토큰화하고, 질문 토큰과 문서 토큰의 BM25 점수로 상위 문서를 반환합니다.


In [ ]:
TOKEN_PATTERN = re.compile(r"[0-9A-Za-z가-힣]+")


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(text.lower())


class BM25Index:
    def __init__(self, documents: list[Document], k1: float = 1.5, b: float = 0.75):
        self.documents = documents
        self.k1 = k1
        self.b = b
        self.doc_tokens = [tokenize(doc.page_content) for doc in documents]
        self.doc_lengths = [len(tokens) for tokens in self.doc_tokens]
        self.avg_doc_length = sum(self.doc_lengths) / max(len(self.doc_lengths), 1)
        self.term_frequencies = [Counter(tokens) for tokens in self.doc_tokens]
        self.document_frequency = Counter()

        for tokens in self.doc_tokens:
            self.document_frequency.update(set(tokens))

        self.document_count = len(documents)

    def idf(self, term: str) -> float:
        df = self.document_frequency.get(term, 0)
        return math.log(1 + (self.document_count - df + 0.5) / (df + 0.5))

    def score(self, query_tokens: list[str], doc_index: int) -> float:
        score = 0.0
        term_frequency = self.term_frequencies[doc_index]
        doc_length = self.doc_lengths[doc_index]

        for term in query_tokens:
            tf = term_frequency.get(term, 0)
            if tf == 0:
                continue
            numerator = tf * (self.k1 + 1)
            denominator = tf + self.k1 * (1 - self.b + self.b * doc_length / self.avg_doc_length)
            score += self.idf(term) * numerator / denominator

        return score

    def search(self, query: str, k: int) -> list[Document]:
        query_tokens = tokenize(query)
        scored = [
            (self.score(query_tokens, index), index)
            for index in range(self.document_count)
        ]
        scored.sort(reverse=True)
        return [self.documents[index] for score, index in scored[:k] if score > 0]


bm25_index = BM25Index(split_documents)


## Vector Search와 Hybrid Search

Hybrid Search는 BM25 결과와 Vector Search 결과를 RRF(Reciprocal Rank Fusion) 방식으로 합칩니다. 즉, 두 검색기에서 상위에 나온 문서를 더 높은 순위로 올립니다.


In [ ]:
def doc_key(doc: Document) -> tuple:
    return (
        doc.metadata.get("source", ""),
        doc.metadata.get("page", ""),
        doc.page_content[:200],
    )


def retrieve_bm25(query: str) -> list[Document]:
    return bm25_index.search(query, k=RETRIEVER_K)


def retrieve_vector(query: str) -> list[Document]:
    return vectorstore.similarity_search(query, k=RETRIEVER_K)


def retrieve_hybrid(query: str) -> list[Document]:
    bm25_docs = bm25_index.search(query, k=HYBRID_CANDIDATES)
    vector_docs = vectorstore.similarity_search(query, k=HYBRID_CANDIDATES)

    fused_scores = {}
    fused_docs = {}

    for docs, weight in [(bm25_docs, 1.0), (vector_docs, 1.0)]:
        for rank, doc in enumerate(docs, start=1):
            key = doc_key(doc)
            fused_docs[key] = doc
            fused_scores[key] = fused_scores.get(key, 0.0) + weight / (60 + rank)

    ranked_keys = sorted(fused_scores, key=fused_scores.get, reverse=True)
    return [fused_docs[key] for key in ranked_keys[:RETRIEVER_K]]


RETRIEVER_CONFIGS = [
    {
        "config_name": "A. BM25",
        "retriever_type": "BM25",
        "retrieve": retrieve_bm25,
        "answers_path": DATA_DIR / "rag_config_answers_bm25.csv",
        "scores_path": DATA_DIR / "rag_config_scores_bm25.csv",
    },
    {
        "config_name": "B. Vector Search",
        "retriever_type": "Vector Search",
        "retrieve": retrieve_vector,
        "answers_path": DATA_DIR / "rag_config_answers_vector.csv",
        "scores_path": DATA_DIR / "rag_config_scores_vector.csv",
    },
    {
        "config_name": "C. Hybrid Search",
        "retriever_type": "BM25 + Vector Search",
        "retrieve": retrieve_hybrid,
        "answers_path": DATA_DIR / "rag_config_answers_hybrid.csv",
        "scores_path": DATA_DIR / "rag_config_scores_hybrid.csv",
    },
]


## 구성별 답변 생성

이 셀은 각 검색기로 실제 문서를 검색하고, 같은 LLM과 같은 프롬프트로 답변을 생성합니다.


In [ ]:
RAG_PROMPT = PromptTemplate.from_template(
    """
당신은 문서 기반 질의응답 어시스턴트입니다.
반드시 아래 컨텍스트에서 확인할 수 있는 내용만 사용해 한국어로 답변하세요.
컨텍스트에 근거가 없으면 "제공된 문서에서는 확인할 수 없습니다."라고 답하세요.

# 컨텍스트
{context}

# 질문
{question}
"""
)

answer_chain = RAG_PROMPT | answer_llm | StrOutputParser()


def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(doc.page_content for doc in docs)


def run_retriever_config(config: dict, question_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    retrieve = config["retrieve"]

    for row in question_df.itertuples(index=False):
        retrieved_docs = retrieve(row.user_input)
        contexts = [doc.page_content for doc in retrieved_docs]
        response = answer_chain.invoke(
            {
                "context": format_docs(retrieved_docs),
                "question": row.user_input,
            }
        )
        rows.append(
            {
                "config_name": config["config_name"],
                "retriever_type": config["retriever_type"],
                "user_input": row.user_input,
                "retrieved_contexts": contexts,
                "response": response,
                "reference": row.reference,
                "reference_contexts": row.reference_contexts,
            }
        )

    return pd.DataFrame(rows)


def save_answer_csv(answer_df: pd.DataFrame, path: Path) -> None:
    output = answer_df.copy()
    for column in ["retrieved_contexts", "reference_contexts"]:
        output[column] = output[column].apply(lambda value: json.dumps(value, ensure_ascii=False))
    output.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"저장 완료: {path}")


In [ ]:
answer_frames = []

for config in RETRIEVER_CONFIGS:
    print(f"답변 생성 중: {config['config_name']}")
    answer_df = run_retriever_config(config, eval_questions)
    save_answer_csv(answer_df, config["answers_path"])
    answer_frames.append(answer_df)

all_answers = pd.concat(answer_frames, ignore_index=True)
all_answers_path = DATA_DIR / "rag_configuration_answers.csv"
save_answer_csv(all_answers, all_answers_path)

display(all_answers[["config_name", "user_input", "response"]].head())


## RAGAS 평가

각 검색기 구성이 만든 답변을 같은 RAGAS 지표로 평가합니다.


In [ ]:
def diagnose_rag_failure(row: pd.Series, threshold: float = 0.70) -> str:
    retrieval_low = min(row["context_recall"], row["context_precision"]) < threshold
    generation_low = min(row["answer_relevancy"], row["faithfulness"]) < threshold

    if retrieval_low and generation_low:
        return "검색과 생성 모두 점검"
    if retrieval_low:
        return "검색 단계 우선 점검"
    if generation_low:
        return "생성 단계 우선 점검"
    return "큰 문제 없음"


def evaluate_answers(answer_df: pd.DataFrame, config: dict) -> pd.DataFrame:
    dataset = Dataset.from_pandas(
        answer_df[["user_input", "retrieved_contexts", "response", "reference", "reference_contexts"]],
        preserve_index=False,
    )
    result = evaluate(
        dataset=dataset,
        metrics=[answer_relevancy, faithfulness, context_recall, context_precision],
        llm=judge_llm,
    )
    score_df = result.to_pandas()
    score_df["config_name"] = config["config_name"]
    score_df["retriever_type"] = config["retriever_type"]
    score_df[METRICS] = score_df[METRICS].apply(pd.to_numeric, errors="coerce")
    score_df["min_score"] = score_df[METRICS].min(axis=1)
    score_df["diagnosis"] = score_df.apply(diagnose_rag_failure, axis=1)
    score_df.to_csv(config["scores_path"], index=False, encoding="utf-8-sig")
    print(f"저장 완료: {config['scores_path']}")
    return score_df


In [ ]:
score_frames = []

for config, answer_df in zip(RETRIEVER_CONFIGS, answer_frames):
    print(f"RAGAS 평가 중: {config['config_name']}")
    score_frames.append(evaluate_answers(answer_df, config))

all_scores = pd.concat(score_frames, ignore_index=True)
scores_path = DATA_DIR / "rag_configuration_scores.csv"
all_scores.to_csv(scores_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {scores_path}")

display(all_scores[["config_name", "user_input", *METRICS, "min_score", "diagnosis"]].head())


## 구성별 평균 점수 비교

검색기 선택은 평균 점수, 낮은 점수 샘플 수, 운영 비용과 지연 시간을 함께 봐야 합니다.


In [ ]:
WEIGHTS = pd.Series(
    {
        "answer_relevancy": 0.25,
        "faithfulness": 0.30,
        "context_recall": 0.25,
        "context_precision": 0.20,
    }
)

all_scores["weighted_score"] = all_scores[METRICS].mul(WEIGHTS).sum(axis=1)
all_scores["is_low_score"] = all_scores["min_score"] < 0.70

summary = (
    all_scores
    .groupby(["config_name", "retriever_type"])
    .agg(
        answer_relevancy=("answer_relevancy", "mean"),
        faithfulness=("faithfulness", "mean"),
        context_recall=("context_recall", "mean"),
        context_precision=("context_precision", "mean"),
        min_score=("min_score", "mean"),
        weighted_score=("weighted_score", "mean"),
        low_score_count=("is_low_score", "sum"),
        sample_count=("user_input", "count"),
    )
    .reset_index()
)

summary["rank"] = summary["weighted_score"].rank(ascending=False, method="dense").astype(int)
summary = summary.sort_values(["rank", "low_score_count", "config_name"])

display(summary.round(4))


In [ ]:
category_summary = (
    all_scores
    .merge(eval_questions[["user_input", "query_category"]], on="user_input", how="left")
    .groupby(["config_name", "query_category"])
    .agg(
        weighted_score=("weighted_score", "mean"),
        min_score=("min_score", "mean"),
        sample_count=("user_input", "count"),
    )
    .reset_index()
    .sort_values(["query_category", "weighted_score"], ascending=[True, False])
)

display(category_summary.round(4))


In [ ]:
low_cases = (
    all_scores
    .loc[all_scores["min_score"] < 0.70, ["config_name", "user_input", *METRICS, "min_score", "diagnosis", "response", "reference"]]
    .sort_values(["config_name", "min_score"])
)

print(f"낮은 점수 샘플 수: {len(low_cases)}")
display(low_cases.head(20))


In [ ]:
summary_path = DATA_DIR / "rag_configuration_summary.csv"
summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {summary_path}")

best_config = summary.iloc[0]
print(f"선택 후보: {best_config['config_name']}")
print(f"가중 평균 점수: {best_config['weighted_score']:.4f}")
print(f"낮은 점수 샘플 수: {int(best_config['low_score_count'])}")


## 실습 과제

1. `COMMON_CHUNK_SIZE`, `COMMON_CHUNK_OVERLAP`, `RETRIEVER_K`를 고정한 상태에서 세 검색기를 실행합니다.
2. `rag_configuration_answers.csv`와 `rag_configuration_scores.csv`를 확인합니다.
3. 전체 평균과 질문군별 점수를 비교합니다.
4. 낮은 점수 샘플을 열어 어떤 검색기가 어떤 질문군에서 약한지 분석합니다.

검색기 비교는 **검색기를 실제로 바꿔 실행한 결과**로만 판단해야 합니다.
